Create Bronze Layer Tables

- stores_bronze
- oil_bronze
- train_bronze (sales)
- holidays_events_bronze
- transactions_bronze

In [0]:
from pyspark.sql.functions import *

In [0]:
# Set up paths
catalog = "store_sales_catalog"
bronze_schema = "bronze"
raw_path = "/Volumes/store_sales_catalog/raw/store_sales_vol"
chk_path = f"{raw_path}/_chk"
schema_path = f"{raw_path}/_schema"

In [0]:
# %sql
# -- Clean Up Tables if needed
# DROP TABLE IF EXISTS store_sales_catalog.bronze.stores_bronze;
# DROP TABLE IF EXISTS store_sales_catalog.bronze.oil_bronze;
# DROP TABLE IF EXISTS store_sales_catalog.bronze.train_bronze ;
# DROP TABLE IF EXISTS store_sales_catalog.bronze.holidays_events_bronze;
# DROP TABLE IF EXISTS store_sales_catalog.bronze.transactions_bronze;

In [0]:
%sql
USE CATALOG store_sales_catalog;
USE SCHEMA bronze;

-- SALES
CREATE TABLE IF NOT EXISTS train_bronze (
  id INT,
  date STRING,
  store_nbr INT,
  family STRING,
  sales DOUBLE,
  onpromotion INT,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (id)
  -- FOREIGN KEY (store_nbr) REFERENCES stores_bronze(store_nbr)
  -- FOREIGN KEY (date) REFERENCES oil_bronze(date)
  -- FOREIGN KEY (date) REFERENCES transactions_bronze(date)
  -- FOREIGN KEY (date) REFERENCES holidays_events_bronze(date)
)
USING DELTA;

-- STORES
CREATE TABLE IF NOT EXISTS stores_bronze (
  store_nbr INT,
  city STRING,
  state STRING,
  type STRING,
  cluster INT,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (store_nbr)
)
USING DELTA;

-- OIL
CREATE TABLE IF NOT EXISTS oil_bronze (
  date STRING,
  dcoilwtico DOUBLE,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (date)
)
USING DELTA;

-- TRANSACTIONS
CREATE TABLE IF NOT EXISTS transactions_bronze (
  date STRING,
  store_nbr INT,
  transactions INT,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (date, store_nbr)
  -- FOREIGN KEY (store_nbr) REFERENCES stores_bronze(store_nbr)
  -- FOREIGN KEY (date) REFERENCES oil_bronze(date)
  -- FOREIGN KEY (date) REFERENCES holidays_events_bronze(date)
)
USING DELTA;

-- HOLIDAYS
CREATE TABLE IF NOT EXISTS holidays_events_bronze (
  date STRING,
  type STRING,
  locale STRING,
  locale_name STRING,
  description STRING,
  transferred STRING,
  _source_file STRING,
  _ingest_ts TIMESTAMP,
  PRIMARY KEY (date, type, locale, locale_name, description)
)
USING DELTA;

Autoloader

In [0]:
# Autoloader custom function
def autoload_csv(input_path, table_name, checkpoint_name, schema_name):
    (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", f"{schema_path}/{schema_name}")
        .option("header", "true")
        .option("inferSchema", "true")
        .load(input_path)
        .withColumn("_source_file", 
            col("_metadata.file_path")
        )
        .withColumn("_ingest_ts", current_timestamp())
        .writeStream
        .trigger(once=True)
        .option("checkpointLocation", f"{chk_path}/{checkpoint_name}")
        .option("mergeSchema", "true")
        .toTable(table_name)
    )

autoload_csv(
    input_path=f"{raw_path}/train",
    table_name=f"{catalog}.{bronze_schema}.train_bronze",
    checkpoint_name="train",
    schema_name="train"
)

autoload_csv(
    input_path=f"{raw_path}/stores",
    table_name=f"{catalog}.{bronze_schema}.stores_bronze",
    checkpoint_name="stores",
    schema_name="stores"
)

autoload_csv(
    input_path=f"{raw_path}/oil",
    table_name=f"{catalog}.{bronze_schema}.oil_bronze",
    checkpoint_name="oil",
    schema_name="oil"
)

autoload_csv(
    input_path=f"{raw_path}/transactions",
    table_name=f"{catalog}.{bronze_schema}.transactions_bronze",
    checkpoint_name="transactions",
    schema_name="transactions"
)

autoload_csv(
    input_path=f"{raw_path}/holidays_events",
    table_name=f"{catalog}.{bronze_schema}.holidays_events_bronze",
    checkpoint_name="holidays_events",
    schema_name="holidays_events"
)


Validation test for Bronze layer

In [0]:
%sql
SELECT COUNT(*), MIN(_ingest_ts), MAX(_ingest_ts)
FROM store_sales_catalog.bronze.train_bronze;

COUNT(*),MIN(_ingest_ts),MAX(_ingest_ts)
3000888,2025-11-28T18:51:05.108Z,2025-11-28T18:51:05.108Z


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.train_bronze LIMIT 10

id,date,store_nbr,family,sales,onpromotion,_source_file,_ingest_ts,_rescued_data
0,2013-01-01,1,AUTOMOTIVE,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
1,2013-01-01,1,BABY CARE,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
2,2013-01-01,1,BEAUTY,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
3,2013-01-01,1,BEVERAGES,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
4,2013-01-01,1,BOOKS,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
5,2013-01-01,1,BREAD/BAKERY,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
6,2013-01-01,1,CELEBRATION,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
7,2013-01-01,1,CLEANING,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
8,2013-01-01,1,DAIRY,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null
9,2013-01-01,1,DELI,0.0,0,/Volumes/store_sales_catalog/raw/store_sales_vol/train/train.csv,2025-11-28T18:51:05.108Z,null


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.oil_bronze LIMIT 10;

date,dcoilwtico,_source_file,_ingest_ts,_rescued_data
2013-01-01,null,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-02,93.14,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-03,92.97,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-04,93.12,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-07,93.2,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-08,93.21,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-09,93.08,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-10,93.81,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-11,93.6,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null
2013-01-14,94.27,/Volumes/store_sales_catalog/raw/store_sales_vol/oil/oil.csv,2025-11-28T18:51:08.879Z,null


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.transactions_bronze LIMIT 10;

date,store_nbr,transactions,_source_file,_ingest_ts,_rescued_data
2013-01-01,25,770,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,1,2111,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,2,2358,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,3,3487,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,4,1922,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,5,1903,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,6,2143,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,7,1874,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,8,3250,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null
2013-01-02,9,2940,/Volumes/store_sales_catalog/raw/store_sales_vol/transactions/transactions.csv,2025-11-28T18:52:12.853Z,null


In [0]:
%sql
SELECT * FROM store_sales_catalog.bronze.holidays_events_bronze LIMIT 10;

date,type,locale,locale_name,description,transferred,_source_file,_ingest_ts,_rescued_data
2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-05-12,Holiday,Local,Puyo,Cantonizacion del Puyo,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-23,Holiday,Local,Guaranda,Cantonizacion de Guaranda,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-25,Holiday,Regional,Imbabura,Provincializacion de Imbabura,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-25,Holiday,Local,Latacunga,Cantonizacion de Latacunga,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
2012-06-25,Holiday,Local,Machala,Fundacion de Machala,False,/Volumes/store_sales_catalog/raw/store_sales_vol/holidays_events/holidays_events.csv,2025-11-28T18:52:15.024Z,null
